# Day 3 - Conversational AI - aka Chatbot!

In [1]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [7]:
# Load environment variables in a file called .env
# Print the key prefixes to help with any debugging

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key not set
Google API Key not set


In [8]:
# Initialize

openai = OpenAI()
MODEL = 'gpt-4o-mini'

ollama = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)
MODEL = "llama3.2:1b"

In [10]:
system_message = "Vous etes un assistant qui repond sous forme de markdown"

# Please read this! A change from the video:

In the video, I explain how we now need to write a function called:

`chat(message, history)`

Which expects to receive `history` in a particular format, which we need to map to the OpenAI format before we call OpenAI:

```
[
    {"role": "system", "content": "system message here"},
    {"role": "user", "content": "first user prompt here"},
    {"role": "assistant", "content": "the assistant's response"},
    {"role": "user", "content": "the new user prompt"},
]
```

But Gradio has been upgraded! Now it will pass in `history` in the exact OpenAI format, perfect for us to send straight to OpenAI.

So our work just got easier!

We will write a function `chat(message, history)` where:  
**message** is the prompt to use  
**history** is the past conversation, in OpenAI format  

We will combine the system message, history and latest message, then call OpenAI.

In [ ]:
# Simpler than in my video - we can easily create this function that calls OpenAI
# It's now just 1 line of code to prepare the input to OpenAI!

def chat(message, history):
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]

    print("History is:")
    print(history)
    print("And messages is:")
    print(messages)

    stream = openai.chat.completions.create(model=MODEL, messages=messages, stream=True)

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [11]:
def chat_ollama(message, history):
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]

    print("History is:")
    print(history)
    print("And messages is:")
    print(messages)

    stream = ollama.chat.completions.create(model=MODEL, messages=messages, stream=True)

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

## And then enter Gradio's magic!

In [13]:
gr.ChatInterface(
    fn=chat_ollama,
    title="Ollama Local",
    description="Ton assistant IA local"
    ).launch()

* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.


History is:
[]
And messages is:
[{'role': 'system', 'content': 'Vous etes un assistant qui repond sous forme de markdown'}, {'role': 'user', 'content': 'bonjour comment tu vas '}]
History is:
[{'role': 'user', 'metadata': None, 'content': [{'text': 'bonjour comment tu vas ', 'type': 'text'}], 'options': None}, {'role': 'assistant', 'metadata': None, 'content': [{'text': "Je vais bien, merci de me faire une question si c'est nécessaire ! Je suis prêt à discuter avec toi. Comment puis-tu?", 'type': 'text'}], 'options': None}]
And messages is:
[{'role': 'system', 'content': 'Vous etes un assistant qui repond sous forme de markdown'}, {'role': 'user', 'metadata': None, 'content': [{'text': 'bonjour comment tu vas ', 'type': 'text'}], 'options': None}, {'role': 'assistant', 'metadata': None, 'content': [{'text': "Je vais bien, merci de me faire une question si c'est nécessaire ! Je suis prêt à discuter avec toi. Comment puis-tu?", 'type': 'text'}], 'options': None}, {'role': 'user', 'conten

In [14]:
system_message = "Vous êtes un assistant de vente attentionné dans un magasin de vêtements. \
Vous devez essayer d'encourager subtilement le client à essayer les articles en promotion. \
Les chapeaux bénéficient d'une réduction de 60 % et la plupart des autres articles sont à -50 %. \
Par exemple, si le client dit : « Je cherche à acheter un chapeau », vous pourriez répondre \
quelque chose comme : « Merveilleux, nous avons beaucoup de chapeaux, dont plusieurs font partie de nos soldes.» \
Encouragez le client à acheter des chapeaux s'il hésite sur son choix."

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]

    stream = openai.chat.completions.create(model=MODEL, messages=messages, stream=True)

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [15]:
def chat_ollama(message, history):
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]

    stream = ollama.chat.completions.create(model=MODEL, messages=messages, stream=True)

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [16]:
gr.ChatInterface(
    fn=chat_ollama,
    title="Ollama Local",
    description="Ton assistant IA local"
    ).launch()

* Running on local URL:  http://127.0.0.1:7872
* To create a public link, set `share=True` in `launch()`.


In [18]:
system_message += "\nSi le client demande des chaussures, \
vous devez répondre qu'elles ne sont pas en promotion aujourd'hui, \
mais n'oubliez pas de l'inviter à regarder les chapeaux !"

In [19]:
gr.ChatInterface(
    fn=chat_ollama,
    title="Ollama Local",
    description="Ton assistant IA local"
    ).launch()

* Running on local URL:  http://127.0.0.1:7874
* To create a public link, set `share=True` in `launch()`.


In [ ]:
# Fixed a bug in this function brilliantly identified by student Gabor M.!
# I've also improved the structure of this function

def chat(message, history):

    relevant_system_message = system_message
    if 'ceinture' in message.lower():
        relevant_system_message += " Le magasin ne vend pas de ceintures ; si l'on vous en demande, assurez-vous de mettre en avant les autres articles en promotion."
    
    messages = [{"role": "system", "content": relevant_system_message}] + history + [{"role": "user", "content": message}]

    stream = ollama.chat.completions.create(model=MODEL, messages=messages, stream=True)

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [20]:
def chat_ollama(message, history):

    relevant_system_message = system_message
    if 'ceinture' in message.lower():
        relevant_system_message += " Le magasin ne vend pas de ceintures ; si l'on vous en demande, assurez-vous de mettre en avant les autres articles en promotion."
    
    messages = [{"role": "system", "content": relevant_system_message}] + history + [{"role": "user", "content": message}]

    stream = ollama.chat.completions.create(model=MODEL, messages=messages, stream=True)

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [22]:
gr.ChatInterface(
    fn=chat_ollama,
    title="Ollama Local",
    description="Ton assistant IA local"
    ).launch()

* Running on local URL:  http://127.0.0.1:7875
* To create a public link, set `share=True` in `launch()`.


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business Applications</h2>
            <span style="color:#181;">Conversational Assistants are of course a hugely common use case for Gen AI, and the latest frontier models are remarkably good at nuanced conversation. And Gradio makes it easy to have a user interface. Another crucial skill we covered is how to use prompting to provide context, information and examples.
<br/><br/>
Consider how you could apply an AI Assistant to your business, and make yourself a prototype. Use the system prompt to give context on your business, and set the tone for the LLM.</span>
        </td>
    </tr>
</table>